# 04 — Aktivasyon Müdahalesi (Activation Intervention)

**Amaç:** Seçilen aktivasyonu kontrollü olarak azaltıp/artırmak ve çıktı olasılığı (output probability) değişimini ölçmek.

In [ ]:
# Temel kütüphaneleri ve kontrollü ölçekleme kancasını içe aktarırız.
import os, sys
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.append(ROOT)
from src.model import build_model
from src.interventions import make_scale_hook
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model(42).to(device)
model.load_state_dict(torch.load('../results/baseline_model.pt', map_location=device))
model.eval()
test = datasets.MNIST('data', train=False, download=True, transform=transforms.ToTensor())
loader = DataLoader(test, batch_size=256, shuffle=False)
selected_neurons = [0, 1, 2]  # 02 numaralı notebooktaki adaylarla değiştirilebilir
factors = [0.0, 0.5, 1.0, 1.5, 2.0]
with torch.no_grad():
    normal_logits = torch.cat([model(x.to(device)).cpu() for x, _ in loader])
normal_prob = torch.softmax(normal_logits, dim=1)
for factor in factors:
    handle = model.net[4].register_forward_hook(make_scale_hook(selected_neurons, factor))
    with torch.no_grad():
        intervened_logits = torch.cat([model(x.to(device)).cpu() for x, _ in loader])
    handle.remove()
    intervened_prob = torch.softmax(intervened_logits, dim=1)
    mean_abs_change = (intervened_prob - normal_prob).abs().mean().item()
    changed_prediction_rate = (intervened_prob.argmax(1) != normal_prob.argmax(1)).float().mean().item()
    print(f'factor={factor:.1f} mean_abs_probability_change={mean_abs_change:.6f} prediction_change_rate={changed_prediction_rate:.6f}')

## Yorum
Faktör=1.0 normal koşuldur. Kontrollü değişiklik ile çıktı olasılığı ve tahmin değişimi birlikte ölçülür. Etkinin farklı örneklerde tekrarlanması ve mekanizma hipoteziyle uyumlu olması, nedensel kanıtı destekleyen değerlendirmenin gücünü artırır.